# Silver — Statistics Monthly Indicators (HPI & Wage Index)
Cleans Bronze HPI and Wage Index data: parses monthly period string into a proper date.

**Source:** `bronze.statistics_iceland_raw` (filtered to `dataset IN ('hpi', 'wage_index')`)\
**Output:** `silver.statistics_monthly_indicators` (Delta table)\
**Period format:** `2024M01` → date = 2024-01-01\
**Date rule:** First day of month

In [ ]:
df_raw = spark.read.table("bronze.statistics_iceland_raw").filter("dataset IN ('hpi', 'wage_index')")
df_raw.createOrReplaceTempView("v_monthly_raw")

In [ ]:
df_silver = spark.sql("""
    WITH transformed AS (
        SELECT
            period_raw,
            dataset AS metric,
            CAST(value_raw AS DOUBLE) AS value,
            TO_DATE(REPLACE(period_raw, 'M', '-'), 'yyyy-MM') AS date,
            ingested_at
        FROM v_monthly_raw
        WHERE value_raw NOT IN ('.', '') AND value_raw IS NOT NULL
    ),
    latest_records AS (
        SELECT 
            period_raw,
            metric,
            value,
            date,
            YEAR(date) AS year,
            MONTH(date) AS month,
            ingested_at,
            ROW_NUMBER() OVER (PARTITION BY date, metric ORDER BY ingested_at DESC) AS rn
        FROM transformed
    )
    SELECT
        period_raw,
        metric,
        year,
        month,
        date,
        value,
        CURRENT_TIMESTAMP() AS processed_at
    FROM latest_records
    WHERE rn = 1
""")

df_silver.createOrReplaceTempView("v_monthly_silver_staged")

if df_silver.isEmpty():
    mssparkutils.notebook.exit("no-data")

In [ ]:
if not spark.catalog.tableExists("silver.statistics_monthly_indicators"):
    df_silver.write.format("delta").saveAsTable("silver.statistics_monthly_indicators")
    print("Created silver.statistics_monthly_indicators.")
else:
    spark.sql("""
        MERGE INTO silver.statistics_monthly_indicators AS target
        USING v_monthly_silver_staged AS source
        ON target.year   = source.year
        AND target.month  = source.month
        AND target.metric = source.metric
        WHEN MATCHED THEN UPDATE SET
            target.value        = source.value,
            target.date         = source.date,
            target.processed_at = source.processed_at
        WHEN NOT MATCHED THEN INSERT (period_raw, metric, year, month, date, value, processed_at)
        VALUES (source.period_raw, source.metric, source.year, source.month, source.date, source.value, source.processed_at)
    """)

print(f"Merge complete. Final count: {spark.table('silver.statistics_monthly_indicators').count()}")